In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# -------------------------------
# LOAD DATA
# -------------------------------
df = pd.read_csv("final_multi_output_dataset.csv")

# -------------------------------
# 1. CLEAN TARGETS
# -------------------------------
target_cols = ["Fatigue", "FutureHealthRisk", "DiabetesRisk", "AnemiaRisk", "PCOSRisk"]

for col in target_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()
    df[col] = df[col].map({
        'low': 0,
        'medium': 1,
        'high': 2
    })

# -------------------------------
# 2. FEATURE ENGINEERING
# -------------------------------

# ---- Gender ----
df['Gender'] = df['Gender'].astype(str).str.strip().str.lower()
df['Gender'] = df['Gender'].map({'male': 0, 'female': 1}).fillna(0)

# ---- BMI Category ----
def bmi_category(bmi):
    if bmi < 18.5:
        return 0
    elif bmi < 25:
        return 1
    elif bmi < 30:
        return 2
    else:
        return 3

df['bmi_category'] = df['BMI'].apply(bmi_category)

# ---- Sleep ----
df['SleepScore'] = 1 - abs(df['SleepHours'] - 7.5) / 7.5

# ---- Digital ----
df['DigitalLoad'] = df['ScreenTime']
df['LateNightImpact'] = df['LateNightUsage'] * df['ScreenTime']

# ---- Activity ----
df['ActivityScore'] = df['ActivityLevel'] / df['ActivityLevel'].max()

df['SedentaryIndex'] = (
    df['SittingTime'] + df['InactivityPeriods']
) / (df['SittingTime'].max() + df['InactivityPeriods'].max())

# ---- Diet ----
df['CalorieBalance'] = df['CalorieIntake'] / 2000
df['MealScore'] = df['MealsPerDay'] / df['MealsPerDay'].max()
df['DietScore'] = df['DietQuality'] / df['DietQuality'].max()

# ---- Stress ----
df['StressScore'] = df['StressLevel'] / df['StressLevel'].max()

# ---- Dopamine ----
df['DopamineScore'] = df['dopamine_score'] / df['dopamine_score'].max()

# -------------------------------
# 3. HANDLE MISSING VALUES
# -------------------------------
df.fillna(df.mean(numeric_only=True), inplace=True)

# -------------------------------
# 4. SPLIT FEATURES & TARGET
# -------------------------------
X = df.drop(columns=target_cols)
y = df[target_cols]

# -------------------------------
# 5. TRAIN TEST SPLIT
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# -------------------------------
# 6. CREATE UNSCALED DATASETS
# -------------------------------
train_unscaled_processed = pd.concat([X_train, y_train], axis=1)
test_unscaled_processed = pd.concat([X_test, y_test], axis=1)

# -------------------------------
# 7. SCALING
# -------------------------------
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

# -------------------------------
# 8. CREATE SCALED DATASETS
# -------------------------------
train_scaled_processed = pd.concat(
    [X_train_scaled, y_train.reset_index(drop=True)], axis=1
)

test_scaled_processed = pd.concat(
    [X_test_scaled, y_test.reset_index(drop=True)], axis=1
)

# -------------------------------
# 9. SAVE ALL 4 DATASETS
# -------------------------------
train_unscaled_processed.to_csv("train_unscaled_processed.csv", index=False)
test_unscaled_processed.to_csv("test_unscaled_processed.csv", index=False)

train_scaled_processed.to_csv("train_scaled_processed.csv", index=False)
test_scaled_processed.to_csv("test_scaled_processed.csv", index=False)

print("✅ All 4 datasets created successfully!")

✅ All 4 datasets created successfully!
